In [ ]:
import torch
import timm  # for model creation
import onnx

# --- Step 1: Create the same model architecture ---
model = timm.create_model('mobilevit_s', pretrained=False, num_classes=4)  # change num_classes if needed

# --- Step 2: Load your trained weights (.pth file) ---
checkpoint = torch.load("D:\git\cv-project-log\disaster\models\mobilevit_s_disaster_II.pth", map_location="cpu")

# Some checkpoints have nested dicts like {'state_dict': ...}
if "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]

model.load_state_dict(checkpoint)
model.eval()

# --- Step 3: Create a dummy input ---
# MobileViT usually takes 224x224 RGB images
dummy_input = torch.randn(1, 3, 224, 224)

# --- Step 4: Export to ONNX ---
torch.onnx.export(
    model,
    dummy_input,
    "mobilevit_s.onnx",
    export_params=True,
    opset_version=14,              # you can use 13 if needed
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
)

print("✅ Successfully converted to mobilevit_s.onnx")

# --- Step 5: Verify ONNX file ---
model_onnx = onnx.load("mobilevit_s.onnx")
onnx.checker.check_model(model_onnx)
print("✅ ONNX model is valid and ready to use.")


In [3]:
import onnx
from onnxsim import simplify

model = onnx.load("D:\git\cv-project-log\mobilevit_s.onnx")
model_simp, check = simplify(model, input_shapes={"input": [1, 3, 224, 224]})
onnx.save(model_simp, "mobilevit_s_fixed.onnx")


WARNING: The argument `input_shapes` is deprecated. Please use `overwrite_input_shapes` and/or `test_input_shapes` 
instead. An error will be raised in the future.